# 01 · Carga y limpieza de datos

**Proyecto semestral IMT2200, Grupo 6:** *¿Mide el NEM lo mismo para todos?*

Este notebook carga las 16 tablas del MINEDUC (4 tipos × procesos 2023-2026),
documenta cada decisión de limpieza y guarda el panel unificado en
`data/processed/panel_completo.parquet`, que es la entrada de `02_eda.ipynb`.
Las funciones están en `src/` y los parámetros en `src/config.py`.

| Decisión | Contenido |
|---|---|
| 1 | Unicidad de `MRUN` en cada tabla |
| 1b | Homologar nombres de columnas entre años |
| 2 | Postulaciones a una fila por estudiante |
| 3 | Panel por año (IP como base, `merge` `left`) |
| 4 | Tipos de datos y no respuesta de ingreso |
| 5 | Egreso reciente y reglas de la brecha |
| 6 | Comparabilidad y concatenación de los 4 años |

In [ ]:
# Configuración: rutas, librerías y módulos del proyecto (carpeta src/).
import sys
from pathlib import Path

# Raíz del proyecto (funciona si el notebook se abre desde notebooks/ o
# desde la raíz) para poder importar los módulos de src/.
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd

from src.carga import cargar_tablas, comparar_columnas, normalizar_columnas
from src.config import (ANIOS, ARCHIVO_PANEL, COLUMNAS_PANEL,
                        ESTADO_SELECCIONADO, ESTADOS_VALIDOS, TABLAS)
from src.limpieza import (a_formato_largo, armar_panel, diferencias_columnas,
                          limpiar_panel, marcar_egreso_reciente,
                          resumir_postulacion, unificar_codigos)

anios = ANIOS
print(f'Raíz del proyecto: {RAIZ}')

## 1. Carga de los datos

Los CSV están en una carpeta de Google Drive y se leen mediante URLs de descarga
directa (`src/config.py`). Con `usar_cache=True`, cada archivo se descarga **una sola
vez** a `data/raw/` y las siguientes ejecuciones lo leen desde el disco. Esa carpeta
está en `.gitignore` para no subir los datos al repositorio.

In [ ]:
# Descarga cada CSV una vez a data/raw/ y luego lo lee desde el disco.
# Con usar_cache=False se lee directo desde Google Drive, sin guardar nada.
tablas = cargar_tablas(anios, usar_cache=True)
ip, sd, ps, m = (tablas[sigla] for sigla in TABLAS)

# Verificación de que las 16 tablas se cargaron (filas x columnas).
for sigla, nombre in TABLAS.items():
    print(f'=== {sigla}: {nombre} ===')
    for a in anios:
        filas, columnas = tablas[sigla][a].shape
        print(f'[OK] {sigla}-{a}: {filas} filas, {columnas} columnas')

## 2. Exploración inicial de columnas (proceso 2026)

Revisión de los nombres reales de las columnas antes de tomar decisiones, para no
asumir nombres (por ejemplo, la electiva de Historia se llama `HCSOC`, no `HCSO`).

In [ ]:
# Columnas reales de cada tabla (proceso 2026), para no asumir nombres.
for sigla in TABLAS:
    print(f'{sigla}: {tablas[sigla][2026].columns.tolist()}')

In [ ]:
# Columnas clave de IP para la brecha (NEM, ranking, dependencia, PAES).
cols_ip = ip[2026].columns.tolist()
claves = ['NEM', 'RANKING', 'DEPEND', 'CIEN', 'HCSO', 'MATE', 'CLEC',
          'REG_ACTUAL']
print('\n'.join(c for c in cols_ip if any(k in c for k in claves)))

In [ ]:
# Columnas de año de egreso y rendición, para el filtro de egreso reciente.
claves_egreso = ['EGRE', 'ANIO', 'ANYO', 'PROMOCION', 'RIND']
print([c for c in cols_ip if any(k in c for k in claves_egreso)])

In [ ]:
# Distribución de ANYO_DE_EGRESO en el proceso 2026 (5  años más recientes).
print(ip[2026]['ANYO_DE_EGRESO'].value_counts().sort_index().tail(5))

## Decisión 1: unicidad de `MRUN` en cada tabla

In [ ]:
# Decisión 1: ver si MRUN es único por tabla en el proceso 2026.
tablas_2026 = {'IP': ip[2026], 'SD': sd[2026], 'PS': ps[2026], 'M': m[2026]}
for etiqueta, tabla in tablas_2026.items():
    dup = tabla['MRUN'].duplicated(keep=False).sum()
    print(f'{etiqueta}: {dup} filas repetidas de {len(tabla)}')

`IP`, `SD` y `M` tienen una fila por estudiante. `PS` tiene una fila por preferencia
(hasta 20 por estudiante), por lo que debe resumirse antes de unir (Decisión 2).

## Decisión 1b: homologar nombres de columnas entre años

Las exploraciones anteriores solo revisaron 2026. Al procesar los otros años se
encontraron diferencias de estructura:

- **PS-2023** viene en formato *ancho* (`ESTADO_PREF_01` … `ESTADO_PREF_20`).
- **IP-2023 y 2024** usan `MATE_*` en algunas columnas de invierno y del proceso
  anterior. No afecta: la brecha usa la rendición regular del proceso actual.
- **M-2023** tiene nombres distintos (`VIA_ADMISION`, `NOMBRE_INSTITUCION`). No
  afecta: de `M` solo se usa `MRUN`.
- **SD-2023** trae columnas adicionales de una encuesta (dispositivos, espacios).

In [ ]:
# Decisión 1b: normaliza los nombres de columnas de las 16 tablas (BOM,
# espacios, mayúsculas) y reporta qué columnas de 2026 faltan o sobran.
for sigla, tabla in tablas.items():
    for a in anios:
        normalizar_columnas(tabla[a])
    comparar_columnas(tabla, sigla)

## Decisión 2: postulaciones a una fila por estudiante

1. Se lleva `PS` a formato largo en los cuatro años (en 2023, las preferencias vacías
   vienen con `ESTADO_PREF = 0` y se eliminan).
2. Se unifica el tipo de `ESTADO_PREF` para que los códigos sean comparables.
3. Se conserva, por estudiante, la preferencia válida mejor ubicada; si no tiene
   ninguna válida, su mejor preferencia inválida.

**Códigos de estado** (inferidos de los conteos; *verificar con el libro de códigos
del DEMRE*): `24` = seleccionado (≈160 mil por año, uno por estudiante, cerca de la
matrícula), `25` = lista de espera, `26` = preferencia eliminada por selección en
una preferencia anterior. El resto son causales de postulación no válida.

In [ ]:
# Decisión 2: PS a formato largo, códigos comparables y una fila por MRUN.
ps_largo = {a: unificar_codigos(a_formato_largo(ps[a], a)) for a in anios}

print(f'\nCódigos usados -> seleccionado: {ESTADO_SELECCIONADO}, '
      f'válidos: {sorted(ESTADOS_VALIDOS)}')
print('Códigos de ESTADO_PREF (conteo por año):')
display(pd.concat({a: ps_largo[a]['ESTADO_PREF'].astype('string')
                   .value_counts() for a in anios}, axis=1).fillna(0)
        .astype(int).head(25))

ps_resumido = {a: resumir_postulacion(ps_largo[a]) for a in anios}
for a in anios:
    n_sel = ps_resumido[a]['seleccionado'].sum()
    print(f'PS {a}: {len(ps_largo[a])} filas -> '
          f'{len(ps_resumido[a])} estudiantes ({n_sel} seleccionados)')

## Decisión 3: panel por año (IP como base, `merge` `left`)

`IP` es el universo completo de inscritos, así que todos los `merge` son `left` para
no perder a quienes se inscribieron pero no postularon, no fueron seleccionados o no
se matricularon: esa comparación es justamente la pregunta de acceso universitario.

In [ ]:
# Decisión 3: un panel por año a nivel estudiante (IP + SD + PS + M).
paneles = {a: armar_panel(ip[a], sd[a], ps_resumido[a], m[a], a)
           for a in anios}
print(f'Panel 2026: {len(paneles[2026])} filas, '
      f'{len(paneles[2026].columns)} columnas')

## Decisión 4: tipos de datos y no respuesta de ingreso

- **Tipos:** al leer IP aparece un `DtypeWarning` en `ANYO_DE_EGRESO`, `RBD` y los
  puntajes: hay celdas en blanco y pandas deja la columna completa como texto. Sin
  corregirlo, filtros como `ANYO_DE_EGRESO == 2025` fallan en silencio. Los valores no
  numéricos pasan a `NaN` y se informa cuántos fueron.
- **Ingreso:** el código 99 de `INGRESO_PERCAPITA_GRUPO_FA` es "no responde", no un
  decil (como pidieron los profesores). Se recodifica a `NaN` sin eliminar las filas:
  en el EDA se mantiene como categoría propia, porque quitarlas sesgaría la muestra.

In [ ]:
# Decisión 4: columnas clave a numérico y código 99 de ingreso -> NaN.
paneles = {a: limpiar_panel(paneles[a], a) for a in anios}
pct_no_responde = paneles[2026]['INGRESO_PERCAPITA_GRUPO_FA'].isna().mean()
print(f'2026: {pct_no_responde:.1%} de no-respuesta en ingreso percápita')

## Decisión 5: egreso reciente y reglas de la brecha

Reglas confirmadas por el equipo:

- **(a) Primera rendición:** la brecha usa `*_REG_ACTUAL` (y `*_INV_ACTUAL` si fue su
  única rendición), nunca `*_MAX`. En los datos, casi nadie rinde en invierno ni en el
  proceso anterior, así que en la práctica es la rendición regular.
- **(b) Electiva:** `CIEN_REG_ACTUAL` o `HCSOC_REG_ACTUAL`; quienes rinden ambas
  forman un grupo propio.
- **(c) Egreso reciente:** solo estudiantes que egresaron de 4° medio el año en que
  rindieron la PAES. El proceso de admisión *t* se rinde a fines de *t - 1*, por lo
  que se usa `ANYO_DE_EGRESO = ANIO - 1` (`EGRESO_OFFSET = 1`).

Aquí solo se marca el egreso reciente. La brecha se calcula **una sola vez**, en
`src/muestra.py`, dentro de la muestra analítica de `02_eda.ipynb`.

In [ ]:
# Decisión 5: marca a los egresados del año de rendición.
paneles = {a: marcar_egreso_reciente(paneles[a], a) for a in anios}
n_reciente = paneles[2026]['egresado_reciente'].sum()
print(f'2026: {n_reciente} de {len(paneles[2026])} son egresados recientes')

## Decisión 6: comparabilidad y concatenación de los 4 años

In [ ]:
# Decisión 6: revisa diferencias de columnas antes de apilar los 4 años.
# Las diferencias esperadas vienen de la Decisión 1b (columnas no usadas).
for anio, diferencia in diferencias_columnas(paneles).items():
    if diferencia:
        print(f'2023 vs {anio}: {len(diferencia)} columnas distintas '
              f'(ej.: {sorted(diferencia)[:5]})')

panel_completo = pd.concat(paneles.values(), ignore_index=True)
print(f'Panel completo: {len(panel_completo)} filas, '
      f'{panel_completo["ANIO"].nunique()} años')

## Guardar el panel

Se guardan solo las columnas que usa el EDA, en formato Parquet (mucho más liviano
y rápido que CSV, y conserva los tipos de datos).

In [ ]:
# Guarda el panel para 02_eda.ipynb (data/processed/ está en .gitignore).
columnas = [c for c in COLUMNAS_PANEL if c in panel_completo.columns]
ARCHIVO_PANEL.parent.mkdir(parents=True, exist_ok=True)
panel_completo[columnas].to_parquet(ARCHIVO_PANEL, index=False)
print(f'Guardado {ARCHIVO_PANEL.name}: {len(panel_completo)} filas, '
      f'{len(columnas)} columnas')